# 외부 데이터 병합 파이프라인

**목적**: 시군구별 사회경제 지표(인구밀도, 고령화율, 1인가구비율, 재정자립도)를 보호동물 데이터와 연계

**연구 질문**:
1. 재정자립도가 낮은 시군구일수록 입양률이 낮은가?
2. 인구밀도가 H3(광역시 vs 도) 기각 결과를 설명하는가?
3. 1인가구비율 증가가 유기동물 발생과 관련 있는가?
4. 고령화율이 높을수록 입양 수요가 낮은가?
5. 다중회귀를 통한 입양률 결정 요인 식별

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# 경로 설정
BASE_DIR = os.path.dirname(os.getcwd())  # 프로젝트 루트
RAW_DIR = os.path.join(BASE_DIR, '01_raw_data')
EXT_DIR = os.path.join(RAW_DIR, 'external')
OUT_DIR = os.path.join(BASE_DIR, '02_outputs', 'data')
FIG_DIR = os.path.join(BASE_DIR, '02_outputs', 'figures')

# 디렉토리 생성
os.makedirs(EXT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print(f"외부 데이터 경로: {EXT_DIR}")
print(f"외부 데이터 파일 목록:")
for f in sorted(os.listdir(EXT_DIR)):
    size = os.path.getsize(os.path.join(EXT_DIR, f)) / 1024
    print(f"  {f} ({size:.1f} KB)")

## 1. 보호동물 데이터 시군구별 집계

먼저 현재 보유한 보호동물 데이터를 시군구 단위로 집계하여 병합 기준 테이블을 생성한다.

In [ ]:
# 보호소 성과 데이터 로드 및 시군구 단위 집계
shelter = pd.read_csv(os.path.join(OUT_DIR, 'agg_shelter_performance.csv'))

# 시군구별 합산
sigungu_animal = shelter.groupby(['시도', '시군구']).agg(
    총보호건수=('총보호건수', 'sum'),
    입양건수=('입양건수', 'sum'),
    자연사건수=('자연사건수', 'sum'),
    안락사건수=('안락사건수', 'sum'),
    반환건수=('반환건수', 'sum'),
    보호소수=('보호소명', 'nunique')
).reset_index()

# 비율 산출
sigungu_animal['입양률'] = (sigungu_animal['입양건수'] / sigungu_animal['총보호건수'] * 100).round(2)
sigungu_animal['안락사율'] = (sigungu_animal['안락사건수'] / sigungu_animal['총보호건수'] * 100).round(2)
sigungu_animal['자연사율'] = (sigungu_animal['자연사건수'] / sigungu_animal['총보호건수'] * 100).round(2)

print(f"시군구 수: {len(sigungu_animal)}")
print(f"시도 수: {sigungu_animal['시도'].nunique()}")
print(f"\n=== 시도별 시군구 수 ===")
print(sigungu_animal.groupby('시도')['시군구'].count().sort_values(ascending=False).to_string())
print(f"\n=== 병합 키 샘플 (시도 + 시군구) ===")
print(sigungu_animal[['시도', '시군구']].head(10).to_string(index=False))

## 2. 행정구역 명칭 매핑 테이블

외부 데이터와 보호동물 데이터의 시군구 명칭을 통일하기 위한 매핑 규칙을 정의한다.

**주요 이슈**:
- 외부 통계: "수원시 장안구, 권선구, 팔달구, 영통구" → 보호동물 데이터: "수원시" (구 단위를 시 단위로 합산)
- 2023년 강원도 → 강원특별자치도, 전라북도 → 전북특별자치도 명칭 변경
- 광역시 구(남구, 동구 등)는 시도+시군구 복합 키로 구분

In [ ]:
# ============================================================
# 시도 명칭 통일 함수
# - 외부 데이터의 시도 명칭을 보호동물 데이터 기준으로 변환
# ============================================================

SIDO_MAPPING = {
    '강원도': '강원특별자치도',
    '전라북도': '전북특별자치도',
    # 나머지는 동일
}

def normalize_sido(sido_name):
    """시도 명칭을 보호동물 데이터 기준으로 통일"""
    sido = sido_name.strip()
    return SIDO_MAPPING.get(sido, sido)


# ============================================================
# 시군구 명칭 정규화 함수
# - "○○시 △△구" → "○○시" (구 단위 → 시 단위 통합)
# - 단, 광역시의 구는 유지 (부산 남구, 대구 동구 등)
# ============================================================

# 구 단위를 시 단위로 통합해야 하는 대상 (도 소속 시)
CITY_WITH_GU = {
    '수원시', '성남시', '안양시', '안산시', '고양시', '용인시', 
    '부천시', '청주시', '천안시', '전주시', '포항시', '창원시',
}

def normalize_sigungu(sido, sigungu):
    """
    시군구 명칭을 보호동물 데이터 기준으로 정규화
    
    규칙:
    1. 도 소속 시의 구 → 시 단위로 합산 (예: "수원시 장안구" → "수원시")
    2. 광역시의 구 → 그대로 유지 (예: 부산 "남구" → "남구")
    3. 제주/세종 특수 처리
    """
    sigungu = sigungu.strip()
    
    # 제주특별자치도: 시군구가 "제주시", "서귀포시"일 수 있음
    # 보호동물 데이터에서는 "제주특별자치도"로 통합된 경우도 있음
    
    # 도 소속 시의 구 단위 → 시 단위 통합
    for city in CITY_WITH_GU:
        if sigungu.startswith(city):
            return city
    
    return sigungu


def preprocess_external_data(df, sido_col='시도', sigungu_col='시군구'):
    """외부 데이터의 행정구역 명칭을 보호동물 데이터 기준으로 정규화"""
    df = df.copy()
    df[sido_col] = df[sido_col].apply(normalize_sido)
    df[sigungu_col] = df.apply(
        lambda row: normalize_sigungu(row[sido_col], row[sigungu_col]), axis=1
    )
    return df


# 테스트
test_cases = [
    ('강원도', '춘천시'),
    ('경기도', '수원시 장안구'),
    ('경기도', '수원시 권선구'),
    ('부산광역시', '남구'),
    ('전라북도', '전주시 완산구'),
]

print("=== 명칭 정규화 테스트 ===")
for sido, sigungu in test_cases:
    norm_sido = normalize_sido(sido)
    norm_sigungu = normalize_sigungu(norm_sido, sigungu)
    print(f"  {sido}/{sigungu} → {norm_sido}/{norm_sigungu}")

## 3. 외부 데이터 로드 및 전처리

아래 셀들은 `01_raw_data/external/` 디렉토리에 외부 데이터 파일이 존재할 때 실행 가능하다.  
파일이 없으면 스킵되며, 수집 가이드에 따라 데이터를 먼저 다운로드해야 한다.

### 3.1 인구 데이터 (주민등록 인구현황)

In [ ]:
# ============================================================
# 3.1 인구 데이터 로드
# 공공데이터포털: 행정안전부_지역별 주민등록 성별 인구현황
# 예상 컬럼: 행정구역(시도), 행정구역(시군구), 총인구수, 세대수
# ============================================================

def load_population_data(ext_dir, years=range(2019, 2025)):
    """
    인구 데이터 로드 및 정규화
    
    파일 형식 (공공데이터포털 기준):
    - 파일명: population_by_region_{연도}.csv 또는 유사
    - 컬럼: 행정기관코드, 행정기관, 총인구수, 세대수 등
    
    주의: 실제 다운로드한 파일의 컬럼명에 따라 아래 코드 수정 필요
    """
    dfs = []
    
    for year in years:
        # 가능한 파일명 패턴
        patterns = [
            f'population_by_region_{year}.csv',
            f'population_{year}.csv',
            f'주민등록인구_{year}.csv',
        ]
        
        filepath = None
        for pattern in patterns:
            candidate = os.path.join(ext_dir, pattern)
            if os.path.exists(candidate):
                filepath = candidate
                break
        
        if filepath is None:
            print(f"  [SKIP] {year}년 인구 데이터 파일 없음")
            continue
        
        df = pd.read_csv(filepath, encoding='utf-8-sig')
        df['연도'] = year
        dfs.append(df)
        print(f"  [OK] {year}년: {len(df)}행 로드")
    
    if not dfs:
        print("\n⚠ 인구 데이터 파일이 없습니다. 수집 가이드를 참조하세요.")
        return None
    
    return pd.concat(dfs, ignore_index=True)

print("=== 인구 데이터 로드 시도 ===")
pop_df = load_population_data(EXT_DIR)

if pop_df is not None:
    print(f"\n로드 완료: {len(pop_df)}행, 컬럼: {pop_df.columns.tolist()}")
    display(pop_df.head())

### 3.2 재정자립도 데이터

In [ ]:
# ============================================================
# 3.2 재정자립도 데이터 로드
# e-지방재정365 또는 KOSIS
# 예상 컬럼: 자치단체명, 재정자립도(%)
# ============================================================

def load_fiscal_data(ext_dir, years=range(2019, 2025)):
    """재정자립도 데이터 로드"""
    dfs = []
    
    for year in years:
        patterns = [
            f'fiscal_independence_{year}.csv',
            f'재정자립도_{year}.csv',
        ]
        
        filepath = None
        for pattern in patterns:
            candidate = os.path.join(ext_dir, pattern)
            if os.path.exists(candidate):
                filepath = candidate
                break
        
        if filepath is None:
            print(f"  [SKIP] {year}년 재정자립도 파일 없음")
            continue
        
        df = pd.read_csv(filepath, encoding='utf-8-sig')
        df['연도'] = year
        dfs.append(df)
        print(f"  [OK] {year}년: {len(df)}행 로드")
    
    if not dfs:
        # 단일 파일 시도 (여러 연도가 한 파일에 포함된 경우)
        single_patterns = ['fiscal_independence.csv', '재정자립도.csv']
        for pattern in single_patterns:
            candidate = os.path.join(ext_dir, pattern)
            if os.path.exists(candidate):
                df = pd.read_csv(candidate, encoding='utf-8-sig')
                print(f"  [OK] 통합 파일: {len(df)}행 로드")
                return df
        
        print("\n⚠ 재정자립도 데이터 파일이 없습니다. 수집 가이드를 참조하세요.")
        return None
    
    return pd.concat(dfs, ignore_index=True)

print("=== 재정자립도 데이터 로드 시도 ===")
fiscal_df = load_fiscal_data(EXT_DIR)

if fiscal_df is not None:
    print(f"\n로드 완료: {len(fiscal_df)}행")
    display(fiscal_df.head())

### 3.3 세대원수별 세대수 (1인가구 비율)

In [ ]:
# ============================================================
# 3.3 세대원수별 세대수 (1인가구 비율 산출)
# 공공데이터포털: 행정안전부_지역별 세대원수별 세대수
# ============================================================

def load_household_data(ext_dir, years=range(2019, 2025)):
    """세대원수별 세대수 데이터 로드"""
    dfs = []
    
    for year in years:
        patterns = [
            f'household_by_size_{year}.csv',
            f'세대원수별세대수_{year}.csv',
        ]
        
        filepath = None
        for pattern in patterns:
            candidate = os.path.join(ext_dir, pattern)
            if os.path.exists(candidate):
                filepath = candidate
                break
        
        if filepath is None:
            print(f"  [SKIP] {year}년 세대 데이터 파일 없음")
            continue
        
        df = pd.read_csv(filepath, encoding='utf-8-sig')
        df['연도'] = year
        dfs.append(df)
        print(f"  [OK] {year}년: {len(df)}행 로드")
    
    if not dfs:
        print("\n⚠ 세대원수별 세대수 데이터가 없습니다. 수집 가이드를 참조하세요.")
        return None
    
    return pd.concat(dfs, ignore_index=True)

print("=== 세대 데이터 로드 시도 ===")
household_df = load_household_data(EXT_DIR)

if household_df is not None:
    print(f"\n로드 완료: {len(household_df)}행")
    display(household_df.head())

## 4. 데이터 병합

외부 데이터를 정규화한 후 보호동물 시군구 집계 데이터와 `시도 + 시군구` 키로 병합한다.

In [ ]:
# ============================================================
# 4. 전체 병합 파이프라인
#
# 실행 조건: 외부 데이터 파일이 01_raw_data/external/에 존재해야 함
# 파일이 없으면 이 셀은 경고만 출력하고 스킵됨
#
# 실제 다운로드 후 컬럼명이 다를 수 있으므로,
# 아래 COLUMN_MAP 딕셔너리를 실제 파일에 맞게 수정해야 함
# ============================================================

def merge_all_data(sigungu_animal, pop_df=None, fiscal_df=None, household_df=None):
    """
    모든 외부 데이터를 보호동물 시군구 데이터와 병합
    
    Parameters:
    -----------
    sigungu_animal : 보호동물 시군구별 집계 (기준 테이블)
    pop_df : 인구 데이터 (총인구, 세대수)
    fiscal_df : 재정자립도 데이터
    household_df : 세대원수별 세대수 데이터
    
    Returns:
    --------
    merged : 병합된 DataFrame
    report : 병합 품질 리포트 (dict)
    """
    merged = sigungu_animal.copy()
    report = {'기준_시군구수': len(merged)}
    
    # --- 인구 데이터 병합 ---
    if pop_df is not None:
        pop_norm = preprocess_external_data(pop_df)
        # 구 단위 → 시 단위 합산 (수치형 컬럼만)
        # TODO: 실제 컬럼명에 맞게 수정
        # pop_agg = pop_norm.groupby(['시도', '시군구']).agg({'총인구수': 'sum', '세대수': 'sum'}).reset_index()
        # merged = merged.merge(pop_agg, on=['시도', '시군구'], how='left')
        # report['인구_매칭률'] = merged['총인구수'].notna().mean()
        print("  [INFO] 인구 데이터 병합 - 컬럼명 확인 후 TODO 해제 필요")
    
    # --- 재정자립도 병합 ---
    if fiscal_df is not None:
        fiscal_norm = preprocess_external_data(fiscal_df)
        # TODO: 실제 컬럼명에 맞게 수정
        # merged = merged.merge(fiscal_norm[['시도', '시군구', '재정자립도']], on=['시도', '시군구'], how='left')
        # report['재정_매칭률'] = merged['재정자립도'].notna().mean()
        print("  [INFO] 재정자립도 병합 - 컬럼명 확인 후 TODO 해제 필요")
    
    # --- 1인가구 비율 병합 ---
    if household_df is not None:
        hh_norm = preprocess_external_data(household_df)
        # TODO: 실제 컬럼명에 맞게 수정
        # hh_norm['1인가구비율'] = hh_norm['1인세대'] / hh_norm['총세대'] * 100
        # merged = merged.merge(hh_norm[['시도', '시군구', '1인가구비율']], on=['시도', '시군구'], how='left')
        print("  [INFO] 1인가구 비율 병합 - 컬럼명 확인 후 TODO 해제 필요")
    
    return merged, report


# 실행
print("=== 데이터 병합 ===")
merged, report = merge_all_data(
    sigungu_animal, 
    pop_df=pop_df if 'pop_df' in dir() and pop_df is not None else None,
    fiscal_df=fiscal_df if 'fiscal_df' in dir() and fiscal_df is not None else None,
    household_df=household_df if 'household_df' in dir() and household_df is not None else None,
)

print(f"\n기준 시군구 수: {report['기준_시군구수']}")
print(f"병합 결과 컬럼: {merged.columns.tolist()}")
print(f"\n외부 데이터가 병합되면 추가될 컬럼: 총인구, 인구밀도, 고령화율, 1인가구비율, 재정자립도")

## 5. 병합 품질 검증

매칭되지 않은 시군구를 확인하고, 명칭 매핑 테이블을 보완한다.

In [ ]:
# ============================================================
# 5. 병합 품질 검증
# - 매칭 실패 시군구 목록 출력
# - 외부 데이터에만 존재하는 시군구 확인
# ============================================================

def validate_merge(merged, external_sigungu_set=None):
    """
    병합 품질 검증
    
    Parameters:
    -----------
    merged : 병합된 DataFrame
    external_sigungu_set : 외부 데이터의 (시도, 시군구) 집합
    """
    print("=== 병합 품질 리포트 ===\n")
    
    # 결측치 현황
    numeric_cols = merged.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        na_count = merged[col].isna().sum()
        if na_count > 0:
            na_pct = na_count / len(merged) * 100
            print(f"  {col}: {na_count}건 결측 ({na_pct:.1f}%)")
    
    # 외부 데이터와의 시군구 비교
    if external_sigungu_set:
        animal_set = set(zip(merged['시도'], merged['시군구']))
        
        # 보호동물에는 있지만 외부 데이터에 없는 시군구
        only_animal = animal_set - external_sigungu_set
        if only_animal:
            print(f"\n  [WARNING] 보호동물에만 존재 ({len(only_animal)}건):")
            for sido, sigungu in sorted(only_animal):
                print(f"    {sido} / {sigungu}")
        
        # 외부 데이터에는 있지만 보호동물 데이터에 없는 시군구
        only_external = external_sigungu_set - animal_set
        if only_external:
            print(f"\n  [INFO] 외부 데이터에만 존재 ({len(only_external)}건):")
            for sido, sigungu in sorted(only_external)[:10]:
                print(f"    {sido} / {sigungu}")
            if len(only_external) > 10:
                print(f"    ... 외 {len(only_external) - 10}건")
    
    print(f"\n  최종 병합 행 수: {len(merged)}")
    print(f"  사용 가능 컬럼: {merged.columns.tolist()}")

# 현재는 외부 데이터 없이 보호동물 데이터만 검증
validate_merge(merged)

## 6. 상관 분석 (외부 데이터 병합 후 실행)

외부 데이터가 병합되면 아래 분석을 수행한다:
1. 사회경제 변수 간 상관행렬
2. 입양률과 각 변수의 산점도 + 추세선
3. 다중회귀 분석 (입양률 결정 요인)

In [ ]:
# ============================================================
# 6.1 상관행렬 히트맵
# ============================================================
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# 분석 대상 변수 (외부 데이터 병합 후 활성화)
ANALYSIS_COLS = ['입양률', '안락사율', '자연사율', '총보호건수', '보호소수']
# 외부 데이터 병합 후 추가될 변수:
# '총인구', '인구밀도', '고령화율', '1인가구비율', '재정자립도'

available_cols = [c for c in ANALYSIS_COLS if c in merged.columns]

if len(available_cols) >= 3:
    corr = merged[available_cols].corr(method='spearman')
    
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
    
    ax.set_xticks(range(len(available_cols)))
    ax.set_yticks(range(len(available_cols)))
    ax.set_xticklabels(available_cols, rotation=45, ha='right')
    ax.set_yticklabels(available_cols)
    
    # 상관계수 표시
    for i in range(len(available_cols)):
        for j in range(len(available_cols)):
            text = ax.text(j, i, f'{corr.iloc[i, j]:.2f}',
                          ha='center', va='center', fontsize=10,
                          color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black')
    
    plt.colorbar(im, ax=ax, label='Spearman 상관계수')
    ax.set_title('시군구별 보호동물 성과 지표 상관행렬\n(외부 데이터 병합 후 변수 추가 예정)', fontsize=14)
    plt.tight_layout()
    
    # 저장
    os.makedirs(os.path.join(FIG_DIR, '외부데이터'), exist_ok=True)
    plt.savefig(os.path.join(FIG_DIR, '외부데이터', '상관행렬_기본.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("저장 완료: 02_outputs/figures/외부데이터/상관행렬_기본.png")
else:
    print("분석 가능한 변수가 부족합니다.")

In [ ]:
# ============================================================
# 6.2 입양률 vs 사회경제 변수 산점도 (외부 데이터 병합 후 실행)
# ============================================================

SOCIOECONOMIC_COLS = ['인구밀도', '고령화율', '1인가구비율', '재정자립도']
available_socio = [c for c in SOCIOECONOMIC_COLS if c in merged.columns]

if available_socio:
    fig, axes = plt.subplots(1, len(available_socio), figsize=(6*len(available_socio), 5))
    if len(available_socio) == 1:
        axes = [axes]
    
    for ax, col in zip(axes, available_socio):
        ax.scatter(merged[col], merged['입양률'], alpha=0.5, s=30, edgecolors='white', linewidth=0.5)
        
        # 추세선
        mask = merged[[col, '입양률']].notna().all(axis=1)
        if mask.sum() > 5:
            from scipy import stats
            slope, intercept, r, p, se = stats.linregress(merged.loc[mask, col], merged.loc[mask, '입양률'])
            x_range = np.linspace(merged[col].min(), merged[col].max(), 100)
            ax.plot(x_range, slope * x_range + intercept, 'r--', linewidth=2, 
                   label=f'r={r:.3f}, p={p:.4f}')
            ax.legend(fontsize=9)
        
        ax.set_xlabel(col, fontsize=12)
        ax.set_ylabel('입양률 (%)', fontsize=12)
        ax.set_title(f'입양률 vs {col}', fontsize=13)
    
    plt.suptitle('시군구별 입양률과 사회경제 변수의 관계', fontsize=15, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '외부데이터', '입양률_vs_사회경제변수.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠ 사회경제 변수가 아직 병합되지 않았습니다.")
    print("  01_raw_data/external/ 디렉토리에 외부 데이터를 넣은 후 셀 3.1~4를 다시 실행하세요.")

## 7. 다중회귀 분석 (외부 데이터 병합 후 실행)

입양률을 종속변수로, 사회경제 변수를 독립변수로 투입하여 어떤 요인이 가장 큰 설명력을 갖는지 분석한다.

In [ ]:
# ============================================================
# 7. 다중회귀 분석
# 종속변수: 입양률
# 독립변수: 인구밀도, 고령화율, 1인가구비율, 재정자립도, 총보호건수
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

TARGET = '입양률'
FEATURES = ['인구밀도', '고령화율', '1인가구비율', '재정자립도', '총보호건수', '보호소수']
available_features = [f for f in FEATURES if f in merged.columns]

if len(available_features) >= 2:
    # 결측치 제거
    analysis_df = merged[[TARGET] + available_features].dropna()
    print(f"분석 대상: {len(analysis_df)}개 시군구, {len(available_features)}개 독립변수\n")
    
    X = analysis_df[available_features]
    y = analysis_df[TARGET]
    
    # 표준화 (계수 비교를 위해)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # 회귀 모델
    model = LinearRegression()
    model.fit(X_scaled, y)
    
    print(f"R² = {model.score(X_scaled, y):.4f}")
    print(f"\n=== 표준화 회귀계수 (영향력 순) ===")
    coef_df = pd.DataFrame({
        '변수': available_features,
        '표준화계수': model.coef_,
        '절대값': np.abs(model.coef_)
    }).sort_values('절대값', ascending=False)
    
    for _, row in coef_df.iterrows():
        direction = "+" if row['표준화계수'] > 0 else "-"
        print(f"  {row['변수']:12s}: {direction}{row['절대값']:.4f}")
    
    # 시각화
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coef_df['표준화계수']]
    ax.barh(coef_df['변수'], coef_df['표준화계수'], color=colors)
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_xlabel('표준화 회귀계수', fontsize=12)
    ax.set_title(f'입양률 결정 요인 분석 (R² = {model.score(X_scaled, y):.3f})', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '외부데이터', '다중회귀_계수.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠ 독립변수가 2개 미만입니다. 외부 데이터 병합 후 실행하세요.")
    print(f"  현재 사용 가능: {available_features}")

## 8. 결과 저장

In [ ]:
# ============================================================
# 8. 병합 결과 저장
# ============================================================

output_path = os.path.join(OUT_DIR, 'merged_sigungu_socioeconomic.csv')
merged.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"저장 완료: {output_path}")
print(f"  행 수: {len(merged)}")
print(f"  컬럼: {merged.columns.tolist()}")
print(f"\n=== 기초 통계 ===")
display(merged.describe().round(2))